# RealVisXL V4.0 — Photorealistic Text-to-Image on Colab

Second attempt: same pipeline as `sdxl_colab.ipynb` but with **RealVisXL V4.0** — a fine-tuned SDXL checkpoint built for photorealism. Base SDXL produced painterly/3D-looking faces; RealVisXL should look like actual photographs.

**Runtime:** GPU (T4, free tier works)

**Model:** [SG161222/RealVisXL_V4.0](https://huggingface.co/SG161222/RealVisXL_V4.0)

## Cell 0 — Persist HuggingFace cache to Google Drive

Mount Drive and redirect HF cache there. **First run** downloads ~12GB to Drive. Every later session loads from Drive in ~30 seconds instead of re-downloading every time.

Make sure your Drive has ~15GB free.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CACHE_DIR = '/content/drive/MyDrive/hf_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ['HF_HOME']            = CACHE_DIR
os.environ['TRANSFORMERS_CACHE'] = CACHE_DIR
os.environ['HF_HUB_CACHE']       = CACHE_DIR
os.environ['TORCH_HOME']         = CACHE_DIR + '/torch'
print(f'HF cache → {CACHE_DIR}')

## Cell 1 — Environment setup

In [ ]:
!pip install -q diffusers==0.30.0 transformers accelerate safetensors invisible_watermark mlflow ftfy
!pip install -q git+https://github.com/openai/CLIP.git

import os
os.makedirs('/content/outputs', exist_ok=True)

import torch
assert torch.cuda.is_available(), 'No GPU detected! Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2 — Load RealVisXL V4.0

In [ ]:
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
import torch

MODEL_ID = 'SG161222/RealVisXL_V4.0'

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
).to('cuda')

# DPM++ SDE Karras is the recommended sampler for RealVisXL
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    use_karras_sigmas=True,
    algorithm_type='sde-dpmsolver++',
)

try:
    pipe.enable_xformers_memory_efficient_attention()
    print('xformers attention enabled')
except Exception as e:
    print(f'xformers not available ({e}), using SDPA fallback')

print(f'{MODEL_ID} loaded')

## Cell 3 — Load CLIP for metrics

In [ ]:
import clip

clip_model, clip_preprocess = clip.load('ViT-B/32', device='cuda')
clip_model.eval()
print('CLIP ViT-B/32 loaded')

## Cell 4 — Metrics (CLIP similarity, sharpness, diversity)

In [ ]:
import numpy as np
from PIL import Image
import torch.nn.functional as F
from scipy.ndimage import convolve


def clip_similarity(pil_image, prompt):
    image_input = clip_preprocess(pil_image).unsqueeze(0).to('cuda')
    text_tokens = clip.tokenize([prompt], truncate=True).to('cuda')
    with torch.no_grad():
        image_features = clip_model.encode_image(image_input)
        text_features = clip_model.encode_text(text_tokens)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
    return (image_features * text_features).sum(dim=-1).item()


def sharpness(pil_image):
    arr = np.array(pil_image.convert('L'), dtype=np.float64)
    kernel = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float64)
    return float(convolve(arr, kernel).var())


def pixel_diversity(pil_image):
    arr = np.array(pil_image, dtype=np.float32) / 255.0
    return float(arr.std())


print('Metrics ready: clip_similarity, sharpness, pixel_diversity')

## Cell 5 — MLflow setup

In [ ]:
import mlflow

# Your laptop's ngrok tunnel — update each session with the fresh ngrok URL
NGROK_URL = 'https://e91a-196-203-208-97.ngrok-free.app'

mlflow.set_tracking_uri(NGROK_URL)
mlflow.set_experiment('realvisxl_phase2')
print(f'MLflow tracking → {NGROK_URL} (your laptop)')

## Cell 6 — Generation function

In [ ]:
import time


def generate(prompt, negative_prompt='', steps=30, guidance=7.5, seed=42,
             width=1024, height=1024):
    torch.cuda.reset_peak_memory_stats()
    gen = torch.Generator('cuda').manual_seed(seed)

    start = time.perf_counter()
    image = pipe(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=gen,
        width=width,
        height=height,
    ).images[0]
    elapsed = time.perf_counter() - start
    vram_peak = torch.cuda.max_memory_allocated() / 1e9

    return image, elapsed, vram_peak


print('generate() ready')

## Cell 7 — Photorealistic portrait prompts + log to MLflow

RealVisXL responds better to photography-flavored prompts (lens, lighting, camera terms). The prompts are tuned accordingly.

Its negative-prompt recipe is also stronger — included below.

In [ ]:
PROMPTS = [
    'RAW photo, portrait of a young woman with red hair, natural skin texture, soft natural lighting, 85mm lens, shallow depth of field, ultra realistic',
    'RAW photo, portrait of an elderly man with white beard and long hair, wrinkles, skin pores, natural lighting, 85mm lens, ultra realistic',
    'RAW photo, portrait of a middle-aged man with blonde hair and blue eyes, natural skin texture, overcast lighting, 85mm lens, ultra realistic',
]

NEGATIVE = (
    '(worst quality, low quality, illustration, 3d, 2d, painting, cartoons, sketch), open mouth, '
    'deformed iris, deformed pupils, extra fingers, fewer fingers, bad anatomy, disfigured, '
    'bad proportions, duplicate, watermark, text, signature'
)

results = []

for prompt in PROMPTS:
    short = prompt.split(',')[1].strip() if ',' in prompt else prompt[:40]
    print(f'\nGenerating: {short}')

    with mlflow.start_run(run_name=short[:40]):
        image, elapsed, vram_peak = generate(
            prompt, negative_prompt=NEGATIVE, steps=30, guidance=4.0, seed=42,
        )

        clip_sim = clip_similarity(image, prompt)
        sharp = sharpness(image)
        diversity = pixel_diversity(image)

        filename = short.replace(' ', '_').replace(',', '')[:60] + '.png'
        path = f'/content/outputs/realvis_{filename}'
        image.save(path)

        mlflow.log_params({
            'prompt': prompt,
            'negative_prompt': NEGATIVE,
            'steps': 30,
            'guidance': 4.0,
            'seed': 42,
            'model': 'RealVisXL-V4.0',
            'scheduler': 'DPM++ SDE Karras',
        })
        mlflow.log_metrics({
            'clip_similarity': clip_sim,
            'sharpness': sharp,
            'pixel_diversity': diversity,
            'generation_time_s': elapsed,
            'vram_peak_gb': vram_peak,
        })
        mlflow.log_artifact(path)

        print(f'  CLIP sim: {clip_sim:.4f}  sharp: {sharp:.1f}  div: {diversity:.3f}  time: {elapsed:.1f}s  vram: {vram_peak:.1f}GB')
        results.append((short, image, clip_sim, sharp, diversity, elapsed))

print(f'\nGenerated {len(results)} images → /content/outputs/')

## Cell 8 — Display generated images

In [ ]:
import matplotlib.pyplot as plt

# Grid view — with titles for quick comparison
fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 6))
if len(results) == 1:
    axes = [axes]
for ax, (label, image, clip_sim, sharp, diversity, elapsed) in zip(axes, results):
    ax.imshow(image)
    ax.axis('off')
    ax.set_title(f'{label}\nCLIP: {clip_sim:.3f}  sharp: {sharp:.0f}  time: {elapsed:.1f}s', fontsize=10)
plt.tight_layout()
plt.show()

# Individual view — clean images, no title — right-click save and send to the deepfake detector
for label, image, clip_sim, sharp, diversity, elapsed in results:
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

## Cell 9 — Parameter sweep on RealVisXL

RealVisXL typically works best at **guidance 3–5** (lower than SDXL Base's 7.5) and **steps 25–40**.
This sweep finds the optimal combo for this model specifically — different from SDXL Base's optimal.

In [ ]:
SWEEP_PROMPT = 'RAW photo, portrait of a young woman with red hair, natural skin texture, soft natural lighting, 85mm lens, shallow depth of field, ultra realistic'

sweep_results = []

for guidance in [3.0, 4.0, 5.0, 7.0]:
    for steps in [25, 30, 40]:
        print(f'guidance={guidance}  steps={steps}')

        with mlflow.start_run(run_name=f'sweep_g{guidance}_s{steps}'):
            image, elapsed, vram_peak = generate(
                SWEEP_PROMPT, negative_prompt=NEGATIVE,
                steps=steps, guidance=guidance, seed=42,
            )
            clip_sim = clip_similarity(image, SWEEP_PROMPT)
            sharp = sharpness(image)
            diversity = pixel_diversity(image)

            filename = f'realvis_sweep_g{guidance}_s{steps}.png'
            path = f'/content/outputs/{filename}'
            image.save(path)

            mlflow.log_params({
                'prompt': SWEEP_PROMPT,
                'steps': steps,
                'guidance': guidance,
                'seed': 42,
                'sweep': True,
                'model': 'RealVisXL-V4.0',
            })
            mlflow.log_metrics({
                'clip_similarity': clip_sim,
                'sharpness': sharp,
                'pixel_diversity': diversity,
                'generation_time_s': elapsed,
            })
            mlflow.log_artifact(path)

            sweep_results.append((guidance, steps, image, clip_sim, sharp))

print(f'\nSweep complete: {len(sweep_results)} combos')

## Cell 10 — Sweep results grid

In [ ]:
import matplotlib.pyplot as plt

guidance_values = sorted(set(g for g, _, _, _, _ in sweep_results))
steps_values = sorted(set(s for _, s, _, _, _ in sweep_results))

fig, axes = plt.subplots(len(guidance_values), len(steps_values),
                         figsize=(4 * len(steps_values), 4 * len(guidance_values)))

lookup = {(g, s): (img, clip_sim, sharp) for g, s, img, clip_sim, sharp in sweep_results}

for i, g in enumerate(guidance_values):
    for j, s in enumerate(steps_values):
        ax = axes[i, j] if len(guidance_values) > 1 else axes[j]
        img, clip_sim, sharp = lookup[(g, s)]
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'g={g}  s={s}\nCLIP: {clip_sim:.3f}  sharp: {sharp:.0f}', fontsize=9)

plt.tight_layout()
plt.show()

## Cell 11 — Launch MLflow UI (optional)

Both experiments (`sdxl_phase1` and `realvisxl_phase2`) share the same SQLite DB, so you can compare side by side in one UI.

In [ ]:
# Not needed anymore — MLflow UI runs on your LAPTOP at http://localhost:5555
# Colab pushes runs to it via the ngrok URL set in Cell 5.
print('Open http://localhost:5555 on your laptop to view runs.')

## ──────────────────────────────────────
## Phase 3 — IP-Adapter (identity lock) + Gemini judge
## ──────────────────────────────────────

Load a real face photo, lock identity via IP-Adapter, apply an edit prompt, judge with Gemini. MLflow is skipped for now.

## Cell 12 — Install IP-Adapter deps

IP-Adapter loader is already in `diffusers` (installed in Cell 1). This cell just makes sure nothing is missing.

In [ ]:
# IP-Adapter is built into diffusers — no separate package needed.
# Weights download happens automatically in Cell 13 from HuggingFace.
print('IP-Adapter support already present in diffusers')

## Cell 13 — Load IP-Adapter on top of RealVisXL

Loads the face-tuned IP-Adapter weights into the existing pipeline. No re-loading of the base model.

In [ ]:
import torch
from transformers import CLIPVisionModelWithProjection

# IP-Adapter "plus-face" uses ViT-H/14 encoder, not SDXL's default CLIP encoder
image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    'h94/IP-Adapter',
    subfolder='models/image_encoder',
    torch_dtype=torch.float16,
).to('cuda')
pipe.image_encoder = image_encoder

pipe.load_ip_adapter(
    'h94/IP-Adapter',
    subfolder='sdxl_models',
    weight_name='ip-adapter-plus-face_sdxl_vit-h.safetensors',
)
pipe.set_ip_adapter_scale(0.7)  # 0.6-0.8 is usually the sweet spot
print('IP-Adapter (face, SDXL) loaded with ViT-H encoder — scale 0.7')

## Cell 14 — Upload reference face photo

Click "Choose files" and upload `brad-pitt.webp` (or any portrait).

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()
REFERENCE_PATH = list(uploaded.keys())[0]

reference_image = Image.open(REFERENCE_PATH).convert('RGB')
reference_image.thumbnail((512, 512))  # IP-Adapter doesn't need high-res reference
display(reference_image)
print(f'Reference loaded: {REFERENCE_PATH}  {reference_image.size}')

## Cell 15 — Generate with identity lock

Same pipeline call as Cell 6's `generate()`, but passes `ip_adapter_image=reference_image`. Identity is conditioned directly in diffusion — no W+ inversion, no optimization.

In [ ]:
import os
import mlflow

# Re-set MLflow in case kernel was reset — point to your laptop's ngrok URL (same as Cell 5)
NGROK_URL = 'https://ef18-196-203-208-97.ngrok-free.app'
mlflow.set_tracking_uri(NGROK_URL)
mlflow.set_experiment('realvisxl_phase2')

# Standard RealVisXL negative prompt (same as Cell 7)
NEGATIVE = (
    '(worst quality, low quality, illustration, 3d, 2d, painting, cartoons, sketch), open mouth, '
    'deformed iris, deformed pupils, extra fingers, fewer fingers, bad anatomy, disfigured, '
    'bad proportions, duplicate, watermark, text, signature'
)


def generate_with_identity(prompt, reference_image, ip_scale=0.7,
                            negative_prompt='', steps=30, guidance=5.0, seed=42,
                            width=1024, height=1024):
    torch.cuda.reset_peak_memory_stats()
    pipe.set_ip_adapter_scale(ip_scale)
    gen = torch.Generator('cuda').manual_seed(seed)
    start = time.perf_counter()
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        ip_adapter_image=reference_image,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=gen,
        width=width,
        height=height,
    ).images[0]
    elapsed = time.perf_counter() - start
    vram_peak = torch.cuda.max_memory_allocated() / 1e9
    return image, elapsed, vram_peak


EDIT_PROMPT = 'RAW photo, portrait of a man with a bald hair and a soft beard, soft natural lighting, 85mm lens, ultra realistic'
IP_SCALE    = 0.7
STEPS       = 30
GUIDANCE    = 5.0
SEED        = 42

with mlflow.start_run(run_name=f'ipa_{EDIT_PROMPT[:30]}'):
    edited_image, elapsed, vram_peak = generate_with_identity(
        EDIT_PROMPT, reference_image,
        ip_scale=IP_SCALE, negative_prompt=NEGATIVE,
        steps=STEPS, guidance=GUIDANCE, seed=SEED,
    )

    # Metrics
    clip_sim  = clip_similarity(edited_image, EDIT_PROMPT)
    sharp     = sharpness(edited_image)
    diversity = pixel_diversity(edited_image)

    # Save artifacts to disk
    os.makedirs('/content/outputs', exist_ok=True)
    output_path = '/content/outputs/ipa_edit.png'
    ref_path    = '/content/outputs/ipa_reference.png'
    edited_image.save(output_path)
    reference_image.save(ref_path)

    # Log to MLflow (laptop via ngrok)
    mlflow.log_params({
        'prompt':          EDIT_PROMPT,
        'negative_prompt': NEGATIVE,
        'ip_scale':        IP_SCALE,
        'steps':           STEPS,
        'guidance':        GUIDANCE,
        'seed':            SEED,
        'model':           'RealVisXL-V4.0',
        'ip_adapter':      'ip-adapter-plus-face_sdxl_vit-h',
        'reference':       REFERENCE_PATH,
        'mode':            'identity_edit',
    })
    mlflow.log_metrics({
        'clip_similarity':   clip_sim,
        'sharpness':         sharp,
        'pixel_diversity':   diversity,
        'generation_time_s': elapsed,
        'vram_peak_gb':      vram_peak,
    })
    mlflow.log_artifact(output_path)
    mlflow.log_artifact(ref_path)

    print(f'Generated in {elapsed:.1f}s | CLIP: {clip_sim:.3f}  sharp: {sharp:.0f}  vram: {vram_peak:.1f}GB')
    print(f'Saved: {output_path}')

# Display
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(reference_image); axes[0].set_title('Reference'); axes[0].axis('off')
axes[1].imshow(edited_image); axes[1].set_title(f'Edit ({elapsed:.1f}s)'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## Cell 16 — Gemini judge (via OpenRouter)

**Setup once:** open the 🔑 icon in Colab's left sidebar → Add secret → name `OPENROUTER_API_KEY`, paste your `sk-or-v1-...` key → toggle Notebook access ON.

Judges identity preservation per feature + whether the edit was applied.

In [ ]:
from google.colab import userdata
import requests, base64, io, json

OPENROUTER_KEY = userdata.get('OPENROUTER_API_KEY')
OPENROUTER_URL = 'https://openrouter.ai/api/v1/chat/completions'
GEMINI_MODEL = 'google/gemini-2.5-flash'


def _pil_to_data_url(pil_image):
    buffer = io.BytesIO()
    pil_image.save(buffer, format='PNG')
    b64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
    return f'data:image/png;base64,{b64}'


def judge_identity(generated_pil, reference_pil, edit_request):
    system_prompt = '''You are a facial identity comparison expert.
Compare the GENERATED face (image 1) against the REFERENCE face (image 2).
Score how well each facial feature's IDENTITY is preserved, and whether the edit was applied.

Respond ONLY with valid JSON, no markdown:
{
    "jawline":      <1-10>,
    "eyes":         <1-10>,
    "nose":         <1-10>,
    "mouth":        <1-10>,
    "hair":         <1-10>,
    "skin_tone":    <1-10>,
    "overall_identity":  <1-10>,
    "edit_success":      <1-10>,
    "verdict":           "<one sentence>"
}'''

    content = [
        {'type': 'image_url', 'image_url': {'url': _pil_to_data_url(generated_pil)}},
        {'type': 'image_url', 'image_url': {'url': _pil_to_data_url(reference_pil)}},
        {'type': 'text', 'text': f'Image 1 is generated, image 2 is reference. Edit request: "{edit_request}". Score identity preservation and edit success.'},
    ]

    resp = requests.post(
        OPENROUTER_URL,
        headers={'Authorization': f'Bearer {OPENROUTER_KEY}', 'Content-Type': 'application/json'},
        json={'model': GEMINI_MODEL,
              'messages': [{'role': 'system', 'content': system_prompt},
                           {'role': 'user', 'content': content}],
              'temperature': 0.1},
        timeout=60,
    )
    if resp.status_code != 200:
        raise RuntimeError(f'OpenRouter {resp.status_code}: {resp.text[:300]}')

    text = resp.json()['choices'][0]['message']['content'].strip()
    if text.startswith('```'):
        text = text.split('\n', 1)[1].rsplit('```', 1)[0].strip()
    return json.loads(text)


scores = judge_identity(edited_image, reference_image, EDIT_PROMPT)
print(json.dumps(scores, indent=2))